# Housing Benefit

Housing Benefit is a means-tested DWP benefit that helps people on a low income with their rent. It is a legacy benefit that is being phased out: new working-age claimants are routed to the Universal Credit housing element instead, and a managed migration of the remaining HB caseload to UC is in progress. Housing Benefit continues to operate for two groups — pensioners and tenants of supported/temporary accommodation — and PolicyEngine models it alongside the UC housing element so reform analyses can move between the two systems cleanly.

Parameters live in `policyengine_uk/parameters/gov/dwp/housing_benefit/` and the formulas in `policyengine_uk/variables/gov/dwp/housing_benefit/`.

## Eligibility

`housing_benefit_eligible` requires all of:

- the benunit was **already claiming** Housing Benefit in the previous period (proxied by a positive `housing_benefit_reported`) — this gates out new working-age claimants per the UC migration policy,
- the benunit is either in **social housing** or eligible for the **Local Housing Allowance** rate (broadly, private renters),
- the benunit is **not claiming Universal Credit**, and
- assessable capital is at or below the working-age (or pension-age) capital limit.

The take-up step is handled separately by the input variable `would_claim_housing_benefit`, populated stochastically when the dataset is built so that aggregates match published DWP caseloads.

## How PolicyEngine computes Housing Benefit

The award is computed in four stages:

1. **Eligible rent** (`benunit_rent`): the contracted rent for the property, capped at the Local Housing Allowance rate where the LHA applies.
2. **Applicable amount** (`housing_benefit_applicable_amount`): a personal allowance derived from the benunit's composition (single, couple, lone parent) and age, plus any disability/carer premiums. This is the "minimum income" the household needs before HB tapers begin.
3. **Means test** (`housing_benefit_entitlement`): the benunit's `housing_benefit_applicable_income` in excess of the applicable amount is tapered against the eligible rent at the published `withdrawal_rate` (currently 0.65). The result is then reduced by `housing_benefit_non_dep_deductions` for any non-dependant adults living in the household.
4. **Benefit cap** (`housing_benefit`): the post-means-test award is finally reduced by `benefit_cap_reduction` if the household's total benefit income exceeds the cap and no cap exemption applies.

The take-up gate (`would_claim_housing_benefit`) is applied inside `housing_benefit_pre_benefit_cap` so the post-cap variable matches reported caseload after take-up.

## Personal allowances

The applicable amount uses age- and household-composition bands. The chart below shows how the weekly allowances have evolved.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from policyengine_uk.system import system
import pandas as pd
import plotly.express as px
from policyengine_core.charts import format_fig

allowances = system.parameters.gov.dwp.housing_benefit.allowances

params = {
    "Single (younger)": allowances.single.younger,
    "Single (older)": allowances.single.older,
    "Single (aged)": allowances.single.aged,
    "Lone parent (younger)": allowances.lone_parent.younger,
    "Lone parent (older)": allowances.lone_parent.older,
    "Lone parent (aged)": allowances.lone_parent.aged,
    "Couple (younger)": allowances.couple.younger,
    "Couple (older)": allowances.couple.older,
    "Couple (aged)": allowances.couple.aged,
}

dates = [f"{y}-04-01" for y in range(2016, 2026)]
rows = [
    {"date": d, "composition": name, "weekly allowance": p(d)}
    for d in dates
    for name, p in params.items()
]
df = pd.DataFrame(rows)

fig = (
    px.line(
        df,
        x="date",
        y="weekly allowance",
        color="composition",
        title="Housing Benefit weekly personal allowances",
    )
    .update_layout(
        yaxis_tickprefix="£",
        yaxis_tickformat=",.2f",
        xaxis_title="Date",
        legend_title="Composition",
    )
    .update_traces(line_shape="hv")
)
fig = format_fig(fig)
fig

## References

- [Social Security Contributions and Benefits Act 1992, s. 130](https://www.legislation.gov.uk/ukpga/1992/4/section/130) — the primary statute creating Housing Benefit.
- [The Housing Benefit Regulations 2006 (SI 2006/213)](https://www.legislation.gov.uk/uksi/2006/213/contents) and [The Housing Benefit (Persons who have attained the qualifying age for state pension credit) Regulations 2006 (SI 2006/214)](https://www.legislation.gov.uk/uksi/2006/214/contents) — the operational regulations for working-age and pension-age claimants respectively.
- DWP, [Housing Benefit: detailed information](https://www.gov.uk/housing-benefit) — user-facing guidance covering the move to Universal Credit.
- DWP, [Housing Benefit caseload statistics](https://www.gov.uk/government/collections/housing-benefit-caseload-statistics) — official caseload and expenditure outturns used by `policyengine-uk-data` for calibration.